# Project 01: Exploratory Data Analysis (EDA) on Titanic

**Internship:** Data Science & AI  
**Level:** Beginner  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn

## Project objective
Explore the Titanic passenger dataset, clean missing values carefully, compare survival rates across passenger groups, and write evidence-based findings.

## Research questions
1. Did women have a higher survival rate than men?
2. Did passenger class relate to survival rate?
3. How did survival rates differ across age groups?

> A survival-rate difference is an association in this dataset. It does not prove that a characteristic caused survival.


In [ ]:
# 1. Import the required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)

print("Libraries imported successfully.")

## 2. Load the Titanic dataset

Seaborn provides a copy of the Titanic dataset. The first run may need an internet connection to download it.

In [ ]:
# Load the dataset
df = sns.load_dataset("titanic")

print("Dataset loaded successfully.")
print("Original dataset shape:", df.shape)

df.head()

## 3. Inspect the dataset

- `shape` shows the number of rows and columns.
- `head()` displays the first five rows.
- `info()` shows column names, data types and non-null counts.
- `describe()` provides descriptive statistics for numeric columns.

In [ ]:
# Dataset dimensions
print("Rows and columns:", df.shape)

# First five records
display(df.head())

# Data types and non-null values
df.info()

# Summary statistics
display(df.describe(include="all").T)

## 4. Understand the important columns

| Column | Meaning |
|---|---|
| `survived` | 0 = did not survive, 1 = survived |
| `pclass` | Passenger ticket class: 1, 2 or 3 |
| `sex` | Passenger sex |
| `age` | Passenger age in years |
| `sibsp` | Number of siblings/spouses aboard |
| `parch` | Number of parents/children aboard |
| `fare` | Passenger fare |
| `embarked` | Port of embarkation |
| `deck` | Cabin deck |

The dataset contains other columns too. We will retain them unless there is a clear reason to remove one.


## 5. Check missing values

Before cleaning, count missing values in each column. This helps us choose a suitable treatment instead of deleting rows without a reason.

In [ ]:
missing_before = df.isna().sum().sort_values(ascending=False)
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)

missing_report = pd.DataFrame({
    "Missing values": missing_before,
    "Missing percentage": missing_percent.round(2)
})

display(missing_report)

## 6. Clean the data with reasons

Cleaning decisions:
- **Age:** Fill missing ages using the median for the same passenger class and sex. Median is less affected by unusually high or low values. If a group has no available median, use the overall age median as a fallback.
- **Embarked:** Fill missing entries with the most frequent port (mode), because it is a categorical column.
- **Deck:** Drop this column because a large proportion of its values are missing, making it unsuitable for this beginner comparison.
- **Other columns:** Keep them; do not remove passenger records just because one field is missing.


In [ ]:
# Work on a copy so the original data remains unchanged
clean_df = df.copy()

# Fill missing age values using the median within each passenger-class/sex group
age_group_medians = clean_df.groupby(["pclass", "sex"])["age"].transform("median")
clean_df["age"] = clean_df["age"].fillna(age_group_medians)

# Fallback for any age values still missing
clean_df["age"] = clean_df["age"].fillna(clean_df["age"].median())

# Fill missing embarked values with the most frequent category
embarked_mode = clean_df["embarked"].mode(dropna=True)[0]
clean_df["embarked"] = clean_df["embarked"].fillna(embarked_mode)

# Drop deck because most entries are missing
if "deck" in clean_df.columns:
    clean_df = clean_df.drop(columns=["deck"])

print("Cleaning completed.")
print("Original shape:", df.shape)
print("Cleaned shape:", clean_df.shape)
print("\nMissing values after cleaning:")
display(clean_df.isna().sum().sort_values(ascending=False).to_frame("Missing values"))

## 7. Check duplicates and data types

We count exact duplicate rows. We do not automatically delete them because an identical-looking row is not necessarily proof of an erroneous passenger record. We also check that the main numeric columns have numeric data types.

In [ ]:
duplicate_count = clean_df.duplicated().sum()
print("Exact duplicate rows:", duplicate_count)

# Check important column data types
display(clean_df.dtypes.to_frame("Data type"))

# Confirm that key numeric columns are numeric
numeric_columns = ["survived", "pclass", "age", "fare"]
print("\nNumeric column checks:")
for column in numeric_columns:
    print(f"{column}: {pd.api.types.is_numeric_dtype(clean_df[column])}")

## 8. Group and compare survival rates

The mean of the `survived` column gives the survival proportion because its values are 0 and 1. Multiply by 100 to express it as a percentage.

In [ ]:
# Survival rate by sex
survival_by_sex = (
    clean_df.groupby("sex", observed=True)["survived"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "survival_rate", "count": "passengers"})
    .sort_values("survival_rate", ascending=False)
)

survival_by_sex["survival_percent"] = survival_by_sex["survival_rate"] * 100
display(survival_by_sex.round(2))

In [ ]:
# Survival rate by passenger class
survival_by_class = (
    clean_df.groupby("pclass", observed=True)["survived"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "survival_rate", "count": "passengers"})
    .sort_values("survival_rate", ascending=False)
)

survival_by_class["survival_percent"] = survival_by_class["survival_rate"] * 100
display(survival_by_class.round(2))

In [ ]:
# Create age groups
age_bins = [0, 12, 18, 40, 60, 100]
age_labels = ["Child (0-12)", "Teen (13-18)", "Adult (19-40)",
              "Middle-aged (41-60)", "Senior (61+)"]

clean_df["age_band"] = pd.cut(
    clean_df["age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

# Survival rate by age group
survival_by_age = (
    clean_df.groupby("age_band", observed=True)["survived"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "survival_rate", "count": "passengers"})
)

survival_by_age["survival_percent"] = survival_by_age["survival_rate"] * 100
display(survival_by_age.round(2))

## 9. Visualize the results

These charts make it easier to compare survival percentages. The charts describe patterns in the dataset; they do not establish cause and effect.

In [ ]:
# Chart 1: Survival rate by sex
plt.figure(figsize=(7, 4))
sns.barplot(data=clean_df, x="sex", y="survived", errorbar=None)
plt.title("Titanic Survival Rate by Sex")
plt.xlabel("Sex")
plt.ylabel("Survival rate")
plt.ylim(0, 1)
plt.show()

In [ ]:
# Chart 2: Survival rate by passenger class
plt.figure(figsize=(7, 4))
sns.barplot(data=clean_df, x="pclass", y="survived", errorbar=None)
plt.title("Titanic Survival Rate by Passenger Class")
plt.xlabel("Passenger class (1 = First, 3 = Third)")
plt.ylabel("Survival rate")
plt.ylim(0, 1)
plt.show()

In [ ]:
# Chart 3: Survival rate by age group
age_chart_data = (
    clean_df.groupby("age_band", observed=True)["survived"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(9, 4))
sns.barplot(data=age_chart_data, x="age_band", y="survived", errorbar=None)
plt.title("Titanic Survival Rate by Age Group")
plt.xlabel("Age group")
plt.ylabel("Survival rate")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 10. Answer the three research questions with actual numbers

The next cell generates the findings directly from the cleaned dataset. This avoids guessing or typing percentages that might not match the data.

In [ ]:
# Convert group means to percentages for readable findings
sex_rates = clean_df.groupby("sex", observed=True)["survived"].mean() * 100
class_rates = clean_df.groupby("pclass", observed=True)["survived"].mean() * 100
age_rates = clean_df.groupby("age_band", observed=True)["survived"].mean() * 100

female_rate = sex_rates.get("female", np.nan)
male_rate = sex_rates.get("male", np.nan)
best_class = class_rates.idxmax()
best_class_rate = class_rates.max()
lowest_class = class_rates.idxmin()
lowest_class_rate = class_rates.min()
best_age_group = age_rates.idxmax()
best_age_rate = age_rates.max()

print("FINDING 1 — SEX")
print(f"Female passengers had a survival rate of {female_rate:.1f}%, compared with {male_rate:.1f}% for male passengers.")

print("\nFINDING 2 — PASSENGER CLASS")
print(f"Passenger class {best_class} had the highest survival rate ({best_class_rate:.1f}%).")
print(f"Passenger class {lowest_class} had the lowest survival rate ({lowest_class_rate:.1f}%).")

print("\nFINDING 3 — AGE GROUP")
print(f"The {best_age_group} group had the highest observed survival rate ({best_age_rate:.1f}%).")

print("\nNote: These are descriptive associations in the Titanic dataset, not proof of causation.")

## 11. Final conclusion

The Titanic dataset shows differences in survival rates across passenger groups. Gender, ticket class and age group are useful variables for exploring these patterns. The analysis uses descriptive statistics and group comparisons; it cannot establish that any one factor caused a passenger to survive.

### Three findings to include in the submission
Use the exact percentages printed in **Section 10**. Copy those three sentences into your internship task submission so the written summary matches your notebook output.

## 12. Final checklist

- [x] Three research questions written before analysis
- [x] Used `head()`, `info()` and `describe()`
- [x] Checked missing values
- [x] Filled age and embarked values with explained methods
- [x] Dropped the mostly-missing deck column with a reason
- [x] Checked duplicate rows and data types
- [x] Compared survival by sex, passenger class and age group
- [x] Created visualizations
- [x] Generated three findings with actual numbers

**Before submitting:** Restart the notebook kernel and choose **Run All**. Confirm that every cell completes without errors, then upload the notebook to GitHub or Google Drive with view access enabled.
